# 2.2 — Averaged top-K factor models

In [ ]:
import sys, pathlib
SRC = pathlib.Path('../../src').resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import statsmodels.api as sm

from paths import CLEAN_DATA, INTERIM_DATA
from metrics import total_return, ann_active_return
from chart import chart


# Load Data

In [ ]:
df_merge = pd.read_csv(CLEAN_DATA / '10y_merged.csv')
df_merge = df_merge.sort_values('date').set_index('date')


In [ ]:
df_factors = pd.read_csv(INTERIM_DATA / '10y_factors.csv')

factors = ['P/E', 'P/B', 'P/S', 'EV/EBITDA', 'FCF Yield', 'Earnings Yield']

df_factors[factors] = df_factors[factors].astype(float)
df_factors[factors] = stats.zscore(df_factors[factors])
zscore = df_factors.drop(columns=['12 Mo Yield', 'Company Id', 'SecId']).copy()
zscore[['P/E', 'P/B', 'P/S', 'EV/EBITDA']] = - zscore[['P/E', 'P/B', 'P/S', 'EV/EBITDA']]


# Train / Test Split

In [ ]:
_n_ = 50

split = int(_n_ / 100 * len(df_merge))

df_train = df_merge.iloc[:split]
df_test  = df_merge.iloc[split:]

bmk_train = df_train['sprtrn']
bmk_test  = df_test['sprtrn']

rf_train = df_train['rf']
rf_test  = df_test['rf']


# Averaged top-K factor models

Run many trials, then average the factor weight vectors of the top-K performing models instead of taking the single best.

In [ ]:
port_size    = 45
trials_train = 10**6
port_train   = 10**3
top_k        = 10**2          # number of top models to average


# Train

best_obj_train = -1e9
best_weight    = None
best_tic       = None
best_ret_train = None

df_m = df_train.copy()
model_list = []            # store (obj, weight) for each trial


for i in range(trials_train):
    
    z_var = zscore.copy()
    
    onelist = np.ones(len(factors))
    weight = np.random.dirichlet(onelist)              # weight vector
    weight = weight / weight.sum()                     # make sure the sum is exactly 1             

    z_var[factors] = z_var[factors].mul(weight, axis=1)
    z_var['score'] = z_var[factors].sum(axis=1)
    
    z_var = z_var.sort_values('score', ascending=False).reset_index(drop=True)

    z_var_tic = z_var['Ticker'].iloc[:port_size]       # top tickers                        
    
    port_ret = df_m[z_var_tic]
    port_ret = port_ret.mean(axis=1)                   # equal weighted monthly avg return

    obj = ann_active_return(port_ret, bmk_train)       # objective function

    # store every model for later averaging
    model_list.append((obj, weight.copy()))

    if obj > best_obj_train:
        best_obj_train = obj
        best_weight    = weight.copy()
        best_tic       = z_var_tic.copy()
        best_ret_train = port_ret.copy()


# average across top K factor models

model_list.sort(key=lambda x: x[0], reverse=True)
top_models  = model_list[:top_k]

avg_weight = np.mean([w for _, w in top_models], axis=0)
avg_weight = avg_weight / avg_weight.sum()      # re normalise just in case

best_weight = avg_weight.copy()                 # overwrite single best with ensemble

# recompute tickers and train return using averaged factor weights
z_var = zscore.copy()
z_var[factors] = z_var[factors].mul(best_weight, axis=1)
z_var['score'] = z_var[factors].sum(axis=1)
z_var = z_var.sort_values('score', ascending=False).reset_index(drop=True)

best_tic = z_var['Ticker'].iloc[:port_size]
port_ret = df_m[best_tic]
best_ret_train = port_ret.mean(axis=1)
best_obj_train = ann_active_return(best_ret_train, bmk_train)


# weight optimization (stock weights on train)

best_obj_w   = -1e9
best_stock_w = None

tickers = list(best_tic)
df_m = df_train.copy()
df_m = df_m[tickers + ['sprtrn']]     

for j in range(port_train):
    
    onelist = np.ones(len(tickers))
    stock_w = np.random.dirichlet(onelist)
    stock_w = stock_w / stock_w.sum()
    
    port_ret = df_m[tickers].mul(stock_w, axis=1).sum(axis=1)

    obj = ann_active_return(port_ret, bmk_train)    
    
    if obj > best_obj_w:
        best_obj_w     = obj
        best_stock_w   = stock_w.copy()
        best_ret_train = port_ret.copy()
        
df_final_train = pd.DataFrame({'Ticker': best_tic, 'Weight': best_stock_w})


#########################################################################  
# Test

df_m_test = df_test.copy()

port_tic = df_m_test[df_final_train['Ticker']].copy()

w_test = df_final_train['Weight'].astype(float).to_numpy()
best_ret_test = port_tic.mul(w_test, axis=1).sum(axis=1)

best_obj_test = ann_active_return(best_ret_test, bmk_test)

#########################################################################  
# final results

print('\nAVG Factor Weights :')
for i, j in zip(factors, best_weight):
    print(f'{i}:', round(j * 100, 2))
    
df_final_train['Weight'] = round(df_final_train['Weight']*100 , 2)
df_final_train = df_final_train.sort_values('Weight', ascending=False).reset_index(drop=True)
display(df_final_train)

print("Best Active Return (Train, after stock weights):", round(best_obj_w, 2), '\n')
chart(best_ret_train, bmk_train, rf_train)

print("Best Active Return (Test):", round(best_obj_test, 2), '\n')
chart(best_ret_test, bmk_test, rf_test)


## Lower-trial variant (smaller search budget)

In [ ]:
port_size    = 30
trials_train = 5*10**3
port_train   = 10**3
top_k        = 10**2          # number of top models to average


# Train

best_obj_train = -1e9
best_weight    = None
best_tic       = None
best_ret_train = None

df_m = df_train.copy()
model_list = []            # store (obj, weight) for each trial


for i in range(trials_train):
    
    z_var = zscore.copy()
    
    onelist = np.ones(len(factors))
    weight = np.random.dirichlet(onelist)              # weight vector
    weight = weight / weight.sum()                     # make sure the sum is exactly 1             

    z_var[factors] = z_var[factors].mul(weight, axis=1)
    z_var['score'] = z_var[factors].sum(axis=1)
    
    z_var = z_var.sort_values('score', ascending=False).reset_index(drop=True)

    z_var_tic = z_var['Ticker'].iloc[:port_size]       # top tickers                        
    
    port_ret = df_m[z_var_tic]
    port_ret = port_ret.mean(axis=1)                   # equal weighted monthly avg return

    obj = ann_active_return(port_ret, bmk_train)       # objective function

    # store every model for later averaging
    model_list.append((obj, weight.copy()))

    if obj > best_obj_train:
        best_obj_train = obj
        best_weight    = weight.copy()
        best_tic       = z_var_tic.copy()
        best_ret_train = port_ret.copy()


# average across top K factor models

model_list.sort(key=lambda x: x[0], reverse=True)
top_models  = model_list[:top_k]

avg_weight = np.mean([w for _, w in top_models], axis=0)
avg_weight = avg_weight / avg_weight.sum()      # re normalise just in case

best_weight = avg_weight.copy()                 # overwrite single best with ensemble

# recompute tickers and train return using averaged factor weights
z_var = zscore.copy()
z_var[factors] = z_var[factors].mul(best_weight, axis=1)
z_var['score'] = z_var[factors].sum(axis=1)
z_var = z_var.sort_values('score', ascending=False).reset_index(drop=True)

best_tic = z_var['Ticker'].iloc[:port_size]
port_ret = df_m[best_tic]
best_ret_train = port_ret.mean(axis=1)
best_obj_train = ann_active_return(best_ret_train, bmk_train)


# weight optimization (stock weights on train)

best_obj_w   = -1e9
best_stock_w = None

tickers = list(best_tic)
df_m = df_train.copy()
df_m = df_m[tickers + ['sprtrn']]     

for j in range(port_train):
    
    onelist = np.ones(len(tickers))
    stock_w = np.random.dirichlet(onelist)
    stock_w = stock_w / stock_w.sum()
    
    port_ret = df_m[tickers].mul(stock_w, axis=1).sum(axis=1)

    obj = ann_active_return(port_ret, bmk_train)    
    
    if obj > best_obj_w:
        best_obj_w     = obj
        best_stock_w   = stock_w.copy()
        best_ret_train = port_ret.copy()
        
df_final_train = pd.DataFrame({'Ticker': best_tic, 'Weight': best_stock_w})


#########################################################################  
# Test

df_m_test = df_test.copy()

port_tic = df_m_test[df_final_train['Ticker']].copy()

w_test = df_final_train['Weight'].astype(float).to_numpy()
best_ret_test = port_tic.mul(w_test, axis=1).sum(axis=1)

best_obj_test = ann_active_return(best_ret_test, bmk_test)

#########################################################################  
# final results

print('\nAVG Factor Weights :')
for i, j in zip(factors, best_weight):
    print(f'{i}:', round(j * 100, 2))
    
df_final_train['Weight'] = round(df_final_train['Weight']*100 , 2)
df_final_train = df_final_train.sort_values('Weight', ascending=False).reset_index(drop=True)
display(df_final_train)

print("Best Active Return (Train, after stock weights):", round(best_obj_w, 2), '\n')
chart(best_ret_train, bmk_train, rf_train)

print("Best Active Return (Test):", round(best_obj_test, 2), '\n')
chart(best_ret_test, bmk_test, rf_test)
